In [ ]:
import numpy as np
from itertools import product
from tqdm import tqdm

# ---------------------------------------------------------------------
# 1)  MAX-CUT SOLVERS
# ---------------------------------------------------------------------
def _cut_weight(W, side):
    """
    Return total weight of edges crossing the cut defined by Boolean mask `side`.
    side[i] = False  ⇔  vertex i in S
    side[i] = True   ⇔  vertex i in T
    """
    # xor outer product → True where vertices reside on opposite sides
    diff = np.logical_xor.outer(side, side)
    return W[diff].sum() / 2          # each edge counted twice in symmetric W


def max_cut_bruteforce(W):
    """
    Exact max-cut via exhaustive search.  First vertex is pinned to 0 to
    avoid the (S,T) ≡ (T,S) symmetry ⇒ 2^(n-1) candidate cuts.
    Returns (cut_value, mask_side).
    """
    n = W.shape[0]
    best_val, best_side = -np.inf, None
    for bits in tqdm(product([0, 1], repeat=n-1)):          # vertex 0 is fixed
        side = np.array((0, *bits), dtype=bool)
        val = _cut_weight(W, side)
        if val > best_val:
            best_val, best_side = val, side
    return best_val, best_side

# ---------------------------------------------------------------------
# 2)  RECURSIVE BISECTION WITH ε THRESHOLD
# ---------------------------------------------------------------------
def recursive_max_cut(W, eps, solver='one_exchange', min_size=2, seed=None):
    """
    Returns a list of index arrays, one per final block.
      * eps      – stop splitting a block if additional cut weight ≤ eps
      * solver   – 'one_exchange' or 'bruteforce'
      * min_size – don't split blocks smaller than this
    """
    rng = np.random.default_rng(seed)
    clusters = [np.arange(W.shape[0])]      # start with the full set
    gains    = {0: np.inf}

    solve = max_cut_bruteforce

    while True:
        # pick the cluster that still promises the biggest gain
        tgt = max(gains, key=gains.get)
        if gains[tgt] <= eps:
            break

        idx = clusters[tgt]
        if idx.size < 2 * min_size:
            # don’t split tiny blocks
            gains[tgt] = -np.inf
            continue

        subW = W[np.ix_(idx, idx)]

        # random warm-start helps the heuristic
        start = rng.integers(0, 2, idx.size, bool) if solver == 'one_exchange' else None
        cut_val, side = solve(subW) if start is None else solve(subW, start=start)

        gain = cut_val
        if gain <= eps:                       # not worth splitting
            gains[tgt] = -np.inf
            continue

        # turn Boolean mask into two new index sets
        S_idx = idx[~side]
        T_idx = idx[side]

        # replace old cluster and append new one
        clusters[tgt] = S_idx
        clusters.append(T_idx)
        gains[tgt]  = gain
        gains[len(clusters) - 1] = gain

    return clusters

In [ ]:
import numpy as np

def discover_blocks_dp(S, k, min_block=1, use_mean=True):
    """
    Partition an ordered sequence into k contiguous blocks using dynamic programming.

    Objective:
        maximize sum of within-block scores

    If S is a similarity matrix, maximizing within-block similarity is equivalent
    to minimizing cross-block similarity up to a constant.

    Parameters
    ----------
    S : np.ndarray, shape (n, n)
        Symmetric similarity matrix.
    k : int
        Number of contiguous blocks.
    min_block : int
        Minimum allowed block length.
    use_mean : bool
        If True, score each block by mean off-diagonal similarity.
        If False, score each block by sum of off-diagonal similarities.

    Returns
    -------
    boundaries : list[tuple[int, int]]
        Block intervals as half-open ranges [(start, end), ...].
    lengths : list[int]
        Block lengths.
    dp_score : float
        Best objective value.
    block_score : np.ndarray, shape (n, n+1)
        Precomputed score for block [i:j], valid when j > i.
    """

    S = np.asarray(S, dtype=float)
    if S.ndim != 2 or S.shape[0] != S.shape[1]:
        raise ValueError("S must be a square matrix")

    n = S.shape[0]
    if not (1 <= k <= n):
        raise ValueError("k must satisfy 1 <= k <= n")
    if min_block < 1:
        raise ValueError("min_block must be >= 1")
    if k * min_block > n:
        raise ValueError("k * min_block cannot exceed n")

    # Ignore diagonal
    S = S.copy()
    np.fill_diagonal(S, 0.0)

    # 2D prefix sum for O(1) rectangle queries
    P = np.zeros((n + 1, n + 1), dtype=float)
    P[1:, 1:] = np.cumsum(np.cumsum(S, axis=0), axis=1)

    def rect_sum(r0, r1, c0, c1):
        """Sum of S[r0:r1, c0:c1]."""
        return P[r1, c1] - P[r0, c1] - P[r1, c0] + P[r0, c0]

    # block_score[i, j] = score of contiguous block [i:j)
    block_score = np.full((n, n + 1), -np.inf, dtype=float)

    for i in range(n):
        for j in range(i + min_block, n + 1):
            m = j - i
            if m == 1:
                # no off-diagonal pairs in a singleton block
                block_score[i, j] = 0.0
                continue

            # Sum over block, excluding diagonal already zeroed.
            # Since S is symmetric, this counts each pair twice.
            total = rect_sum(i, j, i, j)
            pair_sum = total / 2.0
            num_pairs = m * (m - 1) / 2.0

            if use_mean:
                block_score[i, j] = pair_sum / num_pairs
            else:
                block_score[i, j] = pair_sum

    # dp[b, j] = best score using b blocks to cover [0:j)
    dp = np.full((k + 1, n + 1), -np.inf, dtype=float)
    prev = np.full((k + 1, n + 1), -1, dtype=int)
    dp[0, 0] = 0.0

    for b in range(1, k + 1):
        # j is end of current prefix
        for j in range(b * min_block, n + 1):
            # t is start of last block
            t_min = (b - 1) * min_block
            t_max = j - min_block
            best_val = -np.inf
            best_t = -1

            for t in range(t_min, t_max + 1):
                val = dp[b - 1, t] + block_score[t, j]
                if val > best_val:
                    best_val = val
                    best_t = t

            dp[b, j] = best_val
            prev[b, j] = best_t

    # Backtrack
    boundaries = []
    j = n
    for b in range(k, 0, -1):
        t = prev[b, j]
        if t < 0:
            raise RuntimeError("Failed to recover a valid partition")
        boundaries.append((t, j))
        j = t
    boundaries.reverse()

    lengths = [end - start for start, end in boundaries]
    return boundaries, lengths, dp[k, n], block_score

In [ ]:
from __future__ import annotations

from pathlib import Path
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from scipy.linalg import subspace_angles

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst

In [ ]:
# ROI name, should be 3 parts (index.label.category)
target = "09.MF1.F" # "09.MF1.F", "17.AB3.B"
target_parts = target.split(".")
roi_label = f"{int(target_parts[0]):02d}.{target_parts[1]}.{target_parts[2]}"

# load in pre-computed principal angles
pa_local = vst.fetch(f"{pth.SAVEDIR}/dynamic_modes/shifting_subspace/{target}.pkl")
with open(pa_local, "rb") as f:
    pa_saved = pickle.load(f)

def grassman_fn(X):
    if len(X.shape)>3:
        X = np.nanmean(X, axis=0)

    out = np.linalg.norm(np.sin(np.deg2rad(X)), axis=2)
    # out = np.nanmean(X, axis=2)
    return out
    
# get mean principal angle
top = grassman_fn(pa_saved["top"])
all_pa = grassman_fn(pa_saved["all"])
rand = grassman_fn(pa_saved["random"])

fig,axes = plt.subplots(1,3, figsize=(10,3))
sns.heatmap(top, square=True, cbar=True, ax=axes[0])
sns.heatmap(all_pa, square=True, cbar=True, ax=axes[1])
sns.heatmap(rand, square=True, cbar=True, ax=axes[2])
plt.show()

In [ ]:
# ##################### from BRH paper
# total_W = pa.sum() / 2            # undirected graph
# eps = 0.2 * total_W 
# # this takes way too long
# # blocks = recursive_max_cut(pa, eps=eps, min_size=2, solver='bruteforce')
# for k, b in enumerate(blocks):
#     print(f"Block {k}: {b + 1}")

################# approx. max cut with only contiguous blocks
boundaries, lengths, score, _ = discover_blocks_dp(-top, k=2, min_block=1, use_mean=False)

print("boundaries:", boundaries[0])
print("lengths:", lengths)
print("score:", score)

In [ ]:
best, results = select_k_dp(-top, range(1,6), penalty=1000, use_mean=False)
best

In [ ]:
def block_change_scores(D, window=4):
    """
    D: reordered symmetric dissimilarity matrix
    window: number of items to examine on each side of a candidate boundary

    Returns:
        scores: length n-1 array, where scores[i] is boundary score
                between i and i+1
    """
    n = D.shape[0]
    scores = np.full(n - 1, np.nan)

    for i in range(window - 1, n - window):
        left_idx = np.arange(i - window + 1, i + 1)
        right_idx = np.arange(i + 1, i + window + 1)

        left_block = D[np.ix_(left_idx, left_idx)]
        right_block = D[np.ix_(right_idx, right_idx)]
        cross_block = D[np.ix_(left_idx, right_idx)]

        # exclude diagonal from within-block means
        left_mean = left_block[np.triu_indices(window, k=1)].mean()
        right_mean = right_block[np.triu_indices(window, k=1)].mean()
        cross_mean = cross_block.mean()

        scores[i] = cross_mean - 0.5 * (left_mean + right_mean)

    return scores

scores = block_change_scores(top, window=4)
threshold = np.nanmean(scores) + np.nanstd(scores)
boundaries = np.where(scores > threshold)[0]

plt.plot(scores)
plt.axhline(np.nanmean(scores) + np.nanstd(scores), linestyle='--')
plt.xlabel("Boundary index")
plt.ylabel("Change score")
plt.show()

In [ ]:
def dpv2(
    S,
    k,
    min_block=1,
    within_weight=1.0,
    between_weight=1.0,
    use_mean=True,
):
    """
    Partition an ordered sequence into k contiguous blocks using dynamic programming,
    with an objective that rewards high within-block similarity and penalizes
    high similarity between adjacent blocks.

    Objective
    ---------
    For a partition into contiguous blocks B_1, ..., B_k, maximize

        sum_r within_weight * within_score(B_r)
      - sum_r between_weight * between_score(B_{r-1}, B_r)

    where the between-block penalty is only applied to adjacent blocks. This makes
    the DP tractable while still encouraging sharp transitions between neighboring
    temporal states.

    Parameters
    ----------
    S : np.ndarray, shape (n, n)
        Symmetric similarity matrix.
    k : int
        Number of contiguous blocks.
    min_block : int
        Minimum allowed block length.
    within_weight : float
        Weight on within-block similarity.
    between_weight : float
        Weight on between-block similarity penalty.
    use_mean : bool
        If True, normalize within/between terms by number of pairs.
        If False, use raw sums.

    Returns
    -------
    boundaries : list[tuple[int, int]]
        Final block intervals as half-open ranges [(start, end), ...].
    lengths : list[int]
        Block lengths.
    dp_score : float
        Best total score.
    within_score : np.ndarray, shape (n, n+1)
        Precomputed within-block score for [i:j).
    between_score : np.ndarray, shape (n, n+1, n+1)
        Precomputed adjacent-block penalty for [a:b) vs [b:c).
        Only entries with a < b < c are meaningful.
    """
    S = np.asarray(S, dtype=float)
    if S.ndim != 2 or S.shape[0] != S.shape[1]:
        raise ValueError("S must be a square matrix")

    n = S.shape[0]
    if not (1 <= k <= n):
        raise ValueError("k must satisfy 1 <= k <= n")
    if min_block < 1:
        raise ValueError("min_block must be >= 1")
    if k * min_block > n:
        raise ValueError("k * min_block cannot exceed n")

    S = S.copy()
    np.fill_diagonal(S, 0.0)

    # 2D prefix sum for O(1) rectangle sums.
    P = np.zeros((n + 1, n + 1), dtype=float)
    P[1:, 1:] = np.cumsum(np.cumsum(S, axis=0), axis=1)

    def rect_sum(r0, r1, c0, c1):
        return P[r1, c1] - P[r0, c1] - P[r1, c0] + P[r0, c0]

    # within_score[i, j] = score for one block [i:j)
    within_score = np.full((n, n + 1), -np.inf, dtype=float)
    for i in range(n):
        for j in range(i + min_block, n + 1):
            m = j - i
            if m == 1:
                within_score[i, j] = 0.0
                continue

            total = rect_sum(i, j, i, j)
            pair_sum = total / 2.0
            num_pairs = m * (m - 1) / 2.0
            within_score[i, j] = pair_sum / num_pairs if use_mean else pair_sum

    # between_score[a, b, c] = penalty for adjacent blocks [a:b) and [b:c)
    between_score = np.full((n, n + 1, n + 1), np.nan, dtype=float)
    for a in range(n):
        for b in range(a + min_block, n + 1):
            left_len = b - a
            for c in range(b + min_block, n + 1):
                right_len = c - b
                total = rect_sum(a, b, b, c)
                if use_mean:
                    between_score[a, b, c] = total / (left_len * right_len)
                else:
                    between_score[a, b, c] = total

    # dp[b, j, t] = best score for covering [0:j) with b blocks,
    # where the last block is [t:j)
    dp = np.full((k + 1, n + 1, n + 1), -np.inf, dtype=float)
    prev = np.full((k + 1, n + 1, n + 1), -1, dtype=int)

    # base case: one block [0:j)
    for j in range(min_block, n + 1):
        dp[1, j, 0] = within_weight * within_score[0, j]

    for b in range(2, k + 1):
        for j in range(b * min_block, n + 1):
            t_min = (b - 1) * min_block
            t_max = j - min_block

            for t in range(t_min, t_max + 1):
                block_val = within_weight * within_score[t, j]

                best_val = -np.inf
                best_s = -1

                s_min = (b - 2) * min_block
                s_max = t - min_block
                for s in range(s_min, s_max + 1):
                    prev_val = dp[b - 1, t, s]
                    if not np.isfinite(prev_val):
                        continue

                    penalty = between_weight * between_score[s, t, j]
                    val = prev_val + block_val - penalty
                    if val > best_val:
                        best_val = val
                        best_s = s

                dp[b, j, t] = best_val
                prev[b, j, t] = best_s

    # choose best final last-block start
    t_candidates = np.arange((k - 1) * min_block, n - min_block + 1)
    last_scores = dp[k, n, t_candidates]
    best_idx = int(np.nanargmax(last_scores))
    t = int(t_candidates[best_idx])
    best_score = float(last_scores[best_idx])

    # backtrack
    boundaries = []
    j = n
    for b in range(k, 1, -1):
        boundaries.append((t, j))
        s = prev[b, j, t]
        j, t = t, s

    boundaries.append((0, j))
    boundaries.reverse()

    lengths = [end - start for start, end in boundaries]
    return boundaries, lengths, best_score, within_score, between_score

In [ ]:
def select_k_dp(
  S,
  k_values,
  min_block=1,
  use_mean=True,
  penalty=0.0,
):
  results = []

  for k in k_values:
      boundaries, lengths, raw_score, _, _ = dpv2(
          S=S,
          k=k,
          min_block=min_block,
          use_mean=use_mean,
      )
      penalized_score = raw_score - penalty * k
      results.append(
          {
              "k": k,
              "boundaries": boundaries,
              "lengths": lengths,
              "raw_score": raw_score,
              "penalized_score": penalized_score,
          }
      )

  best = max(results, key=lambda d: d["penalized_score"])
  return best, results

In [ ]:
# Example similarity matrix
rng = np.random.default_rng(0)
n = 24
S = rng.normal(0, 0.05, size=(n, n))
S = (S + S.T) / 2

# Create 3 obvious contiguous blocks with higher within-block similarity
true_blocks = [(0, 8), (8, 16), (16, 24)]
for a, b in true_blocks:
    S[a:b, a:b] += 1.0
np.fill_diagonal(S, 1.0)

fig,ax = plt.subplots(1,1)
sns.heatmap(S, square=True, cbar=True, ax=ax)
plt.show()